In [27]:
import sqlite3
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from langgraph.graph.message import MessagesState
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_core.tools import tool
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import interrupt, Command

llm = init_chat_model("openai:gpt-4o-mini")

conn = sqlite3.connect("memory.db", check_same_thread=False)

config = {
    "configurable": {
        "thread_id": "2"
    }
}

# llm.invoke([{"role": "user", "content": "안녕?"}])

In [21]:


class State(MessagesState):
    custom_stuff: str

graph_builder = StateGraph(State)


In [22]:
@tool
def get_human_feedback(poem: str):
    """
    Asks the user for feedback on the poem.
    User this before returning the final response.
    """
    feedback = interrupt(f"Here is the poem, tell me what you think\n{poem}")
    return feedback

llm_with_tools = llm.bind_tools(tools=[get_human_feedback])

def chatbot(state: State):
    response = llm_with_tools.invoke(f"""
        그대는 시를 만드는 전무가 입니다.

        그리고 get human feedback 함수를 시에 대한 피드백을 받기 위해서 사용합니다.

        긍정적인 피드백을 받은 후에만 완성된 시를 반환 할 수 있습니다.

        항상 먼저 피드백을 요청합니다.

        이게 대화 기록입니다.

        {state['messages']}
    """)
    return {
        "messages": [response]
    }

In [23]:
tool_node = ToolNode(
    tools=[get_human_feedback]
)

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)
graph_builder.add_edge("tools", "chatbot")

graph = graph_builder.compile(
    checkpointer=SqliteSaver(conn)
)


In [24]:
result  = graph.invoke(
    {
        'messages': [
        {
            "role": "user",
            "content": "파이썬 코드에 대한 시를 만들어주고 한국말로 작성해줘"
        },
        ]
    },
    config=config
)

In [25]:
for message in result['messages']:
    message.pretty_print()

================================ Human Message =================================

파이썬 코드에 대한 시를 만들어주고 한국말로 작성해줘
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_OHxfSSEkwyPMBkM5awonNKqD)
 Call ID: call_OHxfSSEkwyPMBkM5awonNKqD
  Args:
    poem: 파이썬의 세계, 코드의 춤
변수와 함수가 어우러진 항구,  
우리가 가진 문제들을 해결해 주는,
우아한 알고리즘의 선율.

들여다보면 빛나는 구문,  
파이썬의 여정은 끝이 없고,
포맷된 문자열로 소통하는,
마음과 생각의 다리.

루프의 반복 속에서,  
우리는 함께 단톡처럼 기록하고,
이해와 성장이 흘리는,
열린 코드의 정원.

세상의 모든 지혜가 담긴,  
함수와 모듈들의 페이지에서,  
파이썬, 그대는 우리의 언어,
끝없는 가능성을 품고 싶어.


In [29]:
snapshot = graph.get_state(config)

snapshot.interrupts

()

In [ ]:
response = Command(
    resume="It looks good!"
)

result  = graph.invoke(
    response,
    config=config
)



================================ Human Message =================================

파이썬 코드에 대한 시를 만들어주고 한국말로 작성해줘
================================== Ai Message ==================================
Tool Calls:
  get_human_feedback (call_OHxfSSEkwyPMBkM5awonNKqD)
 Call ID: call_OHxfSSEkwyPMBkM5awonNKqD
  Args:
    poem: 파이썬의 세계, 코드의 춤
변수와 함수가 어우러진 항구,  
우리가 가진 문제들을 해결해 주는,
우아한 알고리즘의 선율.

들여다보면 빛나는 구문,  
파이썬의 여정은 끝이 없고,
포맷된 문자열로 소통하는,
마음과 생각의 다리.

루프의 반복 속에서,  
우리는 함께 단톡처럼 기록하고,
이해와 성장이 흘리는,
열린 코드의 정원.

세상의 모든 지혜가 담긴,  
함수와 모듈들의 페이지에서,  
파이썬, 그대는 우리의 언어,
끝없는 가능성을 품고 싶어.
================================= Tool Message =================================
Name: get_human_feedback

It looks good!
================================== Ai Message ==================================

시를 긍정적으로 평가해 주셔서 감사합니다! 이제 완성된 시를 공유하겠습니다.

---

**파이썬의 세계**

파이썬의 세계, 코드의 춤  
변수와 함수가 어우러진 항구,  
우리가 가진 문제들을 해결해 주는,  
우아한 알고리즘의 선율.  
  
들여다보면 빛나는 구문,  
파이썬의 여정은 끝이 없고,  
포맷된 문자열로 소통하는,  
마음과 생각의 다리.  
  
루프의 반복 속에서,  
우리는 함